# RCC RefCert: guided first run

This notebook is the shortest runnable path through RCC's finite-control model-qualification layer. It shows how a declared quantum process becomes an auditable program semidensity, how its reference gain is checked against code length, and how a sufficient re-encoding can repair a cost shortfall. The examples reproduce selected Appendix F/H constructions and exercise the same kernel available for new models.

Launch Jupyter from the `rcc-refcert` repository root. The environment needs Python 3.10 or newer and NumPy 1.26 or newer. To create a dedicated notebook environment on macOS or Linux, run:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -e ".[notebook]"
python -m ipykernel install --user --name rcc-refcert --display-name "Python (rcc-refcert)"
python -m jupyterlab
```

Select **Python (rcc-refcert)** as the notebook kernel. On Windows, activate `.venv\Scripts\activate` instead. If you only want to inspect the recorded evidence, open [`results/reference_report.md`](results/reference_report.md).

In [ ]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src" / "rcc_refcert").is_dir():
    raise RuntimeError("Launch Jupyter from the rcc-refcert repository root.")
sys.path.insert(0, str(ROOT / "src"))

from rcc_refcert import audit_case, get_case
from rcc_refcert.cli import main

## 1. See the three reference cases

Each case has a distinct job: a transparent demonstration, a nontrivial certificate diagnostic, and a least-fixed-point boundary.

In [ ]:
_ = main(["examples"])

## 2. Run the transparent case

`dephase-or-halt` has a closed-form semidensity. Every supplied certificate route is expected to pass.

In [ ]:
_ = main(["example", "dephase-or-halt"])

## 3. Inspect the program semidensity

The fixed-point output is the terminating program semidensity. Its trace deficit is the nonhalting program mass, and Eq. (H.34) supplies the fixed-model domination constant.

In [ ]:
transparent = audit_case(get_case("dephase-or-halt"))
analysis = transparent.model_analysis
semidensity = analysis.fixed_point.output
if semidensity is None or analysis.fixed_model_domination is None:
    raise RuntimeError("The transparent case did not produce its fixed-model result.")
trace = float(np.trace(semidensity).real)
nonhalting_mass = max(0.0, 1.0 - trace)
print("Program semidensity M:")
print(np.real_if_close(semidensity))
print(f"trace(M): {trace:.12g}")
print(f"nonhalting mass: {nonhalting_mass:.12g}")
print(f"fixed-model C*: {analysis.fixed_model_domination.constant:.12g}")

## 4. Inspect a useful route failure

The multiblock case exercises unequal block dimensions and rectangular Kraus maps. Its original codewords deliberately fail the sufficient condition in Eq. (H.70), exposing a reference-gain cost shortfall; Eq. (H.76) supplies a passing re-encoding.

In [ ]:
_ = main(["example", "multiblock-rectangular", "--details"])

### Locate the H.70 shortfall and the H.76 completion

The following cell exposes the control-wise reference gains, original codewords, and proposed codewords behind the two aggregate sums.

In [ ]:
multiblock = audit_case(get_case("multiblock-rectangular"))
if multiblock.gain_cost is None:
    raise RuntimeError("The multiblock case did not produce its gain-cost report.")
for syntax in multiblock.gain_cost.syntax_reports:
    print(
        f"{syntax.syntax_state}: H.70 sum {syntax.current_weighted_sum:.12g} "
        f"-> H.76 sum {syntax.suggested_weighted_sum:.12g}"
    )
    for action in syntax.actions:
        print(
            f"  {action.action_name}: gain {action.reference_gain:.12g}; "
            f"code {action.current_codeword} -> {action.suggested_codeword}"
        )

## 5. Inspect the nonhalting boundary

Here the full transient spectral radius is one. The least-fixed-point semantics remains valid while the linear inverse reaches its spectral boundary, so Eq. (H.34) awaits the completed semidensity rather than using a finite truncation.

In [ ]:
_ = main(["example", "dark-nonhalting", "--details"])

## 6. Reproduce the complete frozen suite

The final command runs all three cases and the Appendix F witnesses, then compares the normalized suite structure and its report with the checked-in references without overwriting them.

In [ ]:
status = main(["reproduce", "--check"])
if status != 0:
    raise RuntimeError("The frozen reference suite did not match.")

## What the run establishes

These cells establish finite-input and fixed-model numerical statements for declared processes and supplied proof objects. They make selected Appendix F/H constructions inspectable. The associated RCC paper supplies the physical reference-consistency, faithful-transcription (`TC`), and uniform model-family arguments. Continue with [`docs/SCIENTIFIC_SCOPE.md`](docs/SCIENTIFIC_SCOPE.md) for the claim ladder or [`docs/MODEL_GUIDE.md`](docs/MODEL_GUIDE.md) to construct a new finite-control model.